In [1]:
%%writefile /kaggle/working/train_qwen_lora.py
import argparse
import gc
import math
import os
import time
from contextlib import nullcontext
from datetime import timedelta
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
OUT_DIR = Path("/kaggle/working")

CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for i, c in enumerate(CLASSES)}
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 16, alpha: int = 32, dropout: float = 0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        dx = self.dropout(x)
        lora_out = F.linear(F.linear(dx, self.lora_A.to(dx.dtype)), self.lora_B.to(dx.dtype))
        return out + lora_out * self.scaling


def inject_lora(model: nn.Module, targets=LORA_TARGETS, r: int = 16, alpha: int = 32, dropout: float = 0.05) -> int:
    for p in model.parameters():
        p.requires_grad = False
    replaced = 0
    for name, module in list(model.named_modules()):
        leaf = name.split(".")[-1]
        if leaf in targets and isinstance(module, nn.Linear):
            parent = model.get_submodule(".".join(name.split(".")[:-1])) if "." in name else model
            setattr(parent, leaf, LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
            replaced += 1
    return replaced


class QwenClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, num_labels: int = 3, hidden_drop: float = 0.1):
        super().__init__()
        self.backbone = backbone
        h = backbone.config.hidden_size
        self.norm = nn.LayerNorm(h * 2, dtype=torch.float32)
        self.drop = nn.Dropout(hidden_drop)
        self.head = nn.Linear(h * 2, num_labels, dtype=torch.float32)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        hs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state.float()
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (hs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        last_idx = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        last_pool = hs[torch.arange(hs.size(0), device=hs.device), last_idx]
        feat = torch.cat([last_pool, mean_pool], dim=-1)
        return self.head(self.drop(self.norm(feat)))


class LongCausalDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len: int = 1024, labels=None, orig_indices=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
        self.orig_indices = orig_indices if orig_indices is not None else np.arange(len(texts))
        self.eos_id = tokenizer.eos_token_id
        self.sep_ids = tokenizer.encode("\n...\n", add_special_tokens=False) or [self.eos_id]

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        ids = self.tokenizer.encode(self.texts[idx], add_special_tokens=False)
        if not ids:
            ids = [self.eos_id]
        if len(ids) > self.max_len:
            budget = self.max_len - 2 * len(self.sep_ids)
            h = budget // 3
            m = budget // 3
            t = budget - h - m
            mid_start = (len(ids) - m) // 2
            ids = ids[:h] + self.sep_ids + ids[mid_start : mid_start + m] + self.sep_ids + ids[-t:]
        item = {"idx": int(self.orig_indices[idx]), "input_ids": ids}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


def make_collate_fn(pad_id: int, max_len: int):
    def collate_fn(batch):
        max_l = max(len(x["input_ids"]) for x in batch)
        max_l = min(max_len, ((max_l + 7) // 8) * 8)
        bs = len(batch)
        input_ids = torch.full((bs, max_l), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bs, max_l), dtype=torch.long)
        idxs = torch.empty(bs, dtype=torch.long)
        for i, item in enumerate(batch):
            seq = item["input_ids"]
            L = len(seq)
            input_ids[i, :L] = torch.tensor(seq, dtype=torch.long)
            attention_mask[i, :L] = 1
            idxs[i] = item["idx"]
        out = {"idx": idxs, "input_ids": input_ids, "attention_mask": attention_mask}
        if "label" in batch[0]:
            out["labels"] = torch.tensor([x["label"] for x in batch], dtype=torch.long)
        return out

    return collate_fn


def get_trainable_state(module: nn.Module):
    return {k: v.detach().cpu().clone() for k, v in module.named_parameters() if v.requires_grad}


@torch.no_grad()
def predict_ddp(model, loader, n_total: int, device, rank: int, world_size: int):
    model.eval()
    local_idxs, local_logits = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
        local_idxs.append(batch["idx"].numpy())
        local_logits.append(logits.float().cpu().numpy())

    local_idxs = np.concatenate(local_idxs, axis=0)
    local_logits = np.concatenate(local_logits, axis=0)
    gathered = [None] * world_size
    dist.all_gather_object(gathered, (local_idxs, local_logits))
    if rank != 0:
        return None

    full_logits = np.zeros((n_total, len(CLASSES)), dtype=np.float32)
    seen = np.zeros(n_total, dtype=bool)
    for g_idx, g_log in gathered:
        full_logits[g_idx] = g_log
        seen[g_idx] = True
    assert seen.all(), f"Missing predictions for {(~seen).sum()} rows!"
    assert np.isfinite(full_logits).all(), "Non-finite logits detected!"
    return full_logits


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="Qwen/Qwen2.5-1.5B")
    parser.add_argument("--smoke_test", action="store_true")
    parser.add_argument("--max_train_samples", type=int, default=80000)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--eval_batch_size", type=int, default=16)
    parser.add_argument("--grad_accum", type=int, default=4)
    parser.add_argument("--max_len", type=int, default=1024)
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1.5e-4)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--warmup_ratio", type=float, default=0.06)
    parser.add_argument("--evals_per_epoch", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    dist.init_process_group(backend="nccl", timeout=timedelta(minutes=60))
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    set_seed(args.seed + rank)

    # 1. Дані та стратифікована підвибірка (fold 0 не чіпаємо)
    train_df = pd.read_csv(DATA_DIR / "train.csv", usecols=["id", "text", "label", "domain"])
    test_df = pd.read_csv(DATA_DIR / "test.csv", usecols=["id", "text"])
    splits_df = pd.read_parquet(BASE_DIR / "folds.parquet")
    assert (train_df["id"].to_numpy() == splits_df["id"].to_numpy()).all()

    train_df["fold"] = splits_df["fold"].to_numpy()
    train_df["y"] = train_df["label"].map(LABEL2ID).astype(np.int64)

    tr_df = train_df[train_df["fold"] != 0].reset_index(drop=True)
    va_df = train_df[train_df["fold"] == 0].reset_index(drop=True)
    full_va_len, full_te_len = len(va_df), len(test_df)

    target_tr_n = args.max_train_samples if args.max_train_samples > 0 else len(tr_df)
    target_tr_n = min(target_tr_n, len(tr_df))

    if args.smoke_test:
        tr_df = tr_df.sample(n=1000, random_state=args.seed).reset_index(drop=True)
        va_df = va_df.sample(n=500, random_state=args.seed).reset_index(drop=True)
        test_df = test_df.head(500).reset_index(drop=True)
    elif target_tr_n < len(tr_df):
        strata = tr_df["domain"].astype(str) + "_" + tr_df["label"].astype(str)
        if strata.value_counts().min() < 2:
            strata = tr_df["label"]
        tr_idx, _ = train_test_split(
            np.arange(len(tr_df)), train_size=target_tr_n, stratify=strata, random_state=args.seed
        )
        tr_df = tr_df.iloc[np.sort(tr_idx)].reset_index(drop=True)

    # 2. Токенізатор і датасети (очищуємо RAM одразу після створення)
    if rank != 0:
        dist.barrier()
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    if rank == 0:
        dist.barrier()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    collate_fn = make_collate_fn(tokenizer.pad_token_id, args.max_len)

    va_order = np.argsort(va_df["text"].str.len().to_numpy())
    te_order = np.argsort(test_df["text"].str.len().to_numpy())

    tr_ds = LongCausalDataset(tr_df["text"].tolist(), tokenizer, args.max_len, tr_df["y"].to_numpy())
    va_ds = LongCausalDataset(va_df["text"].iloc[va_order].tolist(), tokenizer, args.max_len, None, orig_indices=va_order)
    te_ds = LongCausalDataset(test_df["text"].iloc[te_order].tolist(), tokenizer, args.max_len, None, orig_indices=te_order)

    va_ids, va_y = va_df["id"].to_numpy(), va_df["y"].to_numpy()
    te_ids = test_df["id"].to_numpy()
    del train_df, splits_df, tr_df, va_df, test_df
    gc.collect()

    tr_sampler = DistributedSampler(tr_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=args.seed)
    va_sampler = DistributedSampler(va_ds, num_replicas=world_size, rank=rank, shuffle=False)
    te_sampler = DistributedSampler(te_ds, num_replicas=world_size, rank=rank, shuffle=False)

    tr_loader = DataLoader(tr_ds, batch_size=args.batch_size, sampler=tr_sampler, num_workers=0, pin_memory=True, collate_fn=collate_fn)
    va_loader = DataLoader(va_ds, batch_size=args.eval_batch_size, sampler=va_sampler, num_workers=0, pin_memory=True, collate_fn=collate_fn)
    te_loader = DataLoader(te_ds, batch_size=args.eval_batch_size, sampler=te_sampler, num_workers=0, pin_memory=True, collate_fn=collate_fn)

    # 3. Модель (база у FP16, LoRA + голова у FP32)
    if rank != 0:
        dist.barrier()
    backbone = AutoModel.from_pretrained(args.model_name, torch_dtype=torch.float16, attn_implementation="sdpa")
    if rank == 0:
        dist.barrier()

    n_lora_layers = inject_lora(backbone, r=args.lora_r, alpha=args.lora_alpha, dropout=0.05)
    backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    backbone.enable_input_require_grads()

    model = QwenClassifier(backbone, num_labels=len(CLASSES)).to(device)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    assert all(p.dtype == torch.float32 for p in trainable_params), "Trainable params must be FP32!"

    if rank == 0:
        n_tr_p = sum(p.numel() for p in trainable_params)
        n_all_p = sum(p.numel() for p in model.parameters())
        mode = "SMOKE TEST" if args.smoke_test else "FULL RUN"
        eff_bs = args.batch_size * args.grad_accum * world_size
        print(
            f"[{mode}] {args.model_name} | LoRA layers={n_lora_layers} (r={args.lora_r}) | "
            f"Trainable={n_tr_p/1e6:.2f}M / {n_all_p/1e6:.1f}M ({n_tr_p/n_all_p*100:.2f}%) | "
            f"max_len={args.max_len} | eff_bs={eff_bs} | lr={args.lr}",
            flush=True,
        )
        print(f"Rows -> Train: {len(tr_ds)} | Val: {len(va_ds)} | Test: {len(te_ds)}", flush=True)

    model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

    no_decay = ("bias", "norm.weight", "norm.bias")
    grouped_params = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)], "weight_decay": args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(grouped_params, lr=args.lr, eps=1e-6)

    steps_per_epoch = math.ceil(len(tr_loader) / args.grad_accum)
    total_opt_steps = max(1, int(math.ceil(steps_per_epoch * args.epochs)))
    warmup_steps = int(total_opt_steps * args.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_opt_steps)

    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", init_scale=4096.0)

    eval_interval = max(1, steps_per_epoch // args.evals_per_epoch)
    best_f1 = -1.0
    best_oof_logits = None
    best_ckpt_path = OUT_DIR / ("qwen15b_lora_smoke.pt" if args.smoke_test else "qwen15b_lora_adapter.pt")

    # 4. Тренування
    global_opt_step = 0
    nan_skips = 0
    total_eval_time = 0.0
    last_eval_time = 0.0
    t_train_start = time.time()

    for epoch in range(math.ceil(args.epochs)):
        tr_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss, log_steps = 0.0, 0
        t_log = time.time()

        for step, batch in enumerate(tr_loader, start=1):
            is_sync = (step % args.grad_accum == 0) or (step == len(tr_loader))
            ctx = nullcontext() if is_sync else model.no_sync()

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with ctx:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(logits.float(), labels) / args.grad_accum
                scaler.scale(loss).backward()

            running_loss += loss.item() * args.grad_accum
            log_steps += 1

            if is_sync:
                scale_before = scaler.get_scale()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if scaler.get_scale() < scale_before:
                    nan_skips += 1
                else:
                    scheduler.step()

                global_opt_step += 1

                if rank == 0 and (global_opt_step % max(1, min(100, eval_interval)) == 0 or global_opt_step == 10):
                    dt = time.time() - t_log
                    vram = torch.cuda.max_memory_allocated(device) / 1024**3
                    print(
                        f"Ep {epoch + 1} | step {global_opt_step}/{total_opt_steps} | "
                        f"loss={running_loss / log_steps:.4f} | lr={scheduler.get_last_lr()[0]:.2e} | "
                        f"vram_peak={vram:.2f}GB | skips={nan_skips} | {dt:.1f}s",
                        flush=True,
                    )
                    running_loss, log_steps, t_log = 0.0, 0, time.time()

                do_eval = (global_opt_step % eval_interval == 0) or (global_opt_step >= total_opt_steps)
                if do_eval:
                    t_ev = time.time()
                    val_logits = predict_ddp(model, va_loader, len(va_ds), device, rank, world_size)
                    last_eval_time = time.time() - t_ev
                    total_eval_time += last_eval_time
                    if rank == 0:
                        val_preds = val_logits.argmax(axis=1)
                        macro_f1 = f1_score(va_y, val_preds, average="macro")
                        per_cls = f1_score(va_y, val_preds, average=None, labels=[0, 1, 2])
                        print(
                            f">>> [EVAL step {global_opt_step}/{total_opt_steps}] "
                            f"Macro-F1={macro_f1:.4f} (H={per_cls[0]:.4f}, F={per_cls[1]:.4f}, A={per_cls[2]:.4f}) "
                            f"| eval_time={last_eval_time:.1f}s",
                            flush=True,
                        )
                        if macro_f1 > best_f1:
                            best_f1 = macro_f1
                            best_oof_logits = val_logits.copy()
                            torch.save(get_trainable_state(model.module), best_ckpt_path)
                    dist.barrier()
                    model.train()

                if global_opt_step >= total_opt_steps:
                    break

    pure_train_sec = (time.time() - t_train_start) - total_eval_time

    # 5. Інференс найкращого адаптера на test
    dist.barrier()
    state = torch.load(best_ckpt_path, map_location=device, weights_only=True)
    model.module.load_state_dict(state, strict=False)
    t_te = time.time()
    te_logits = predict_ddp(model, te_loader, len(te_ds), device, rank, world_size)
    te_elapsed = time.time() - t_te

    if rank == 0:
        vram_peak = torch.cuda.max_memory_allocated(device) / 1024**3
        print(f"\n=== FINISHED | Best Val Macro-F1={best_f1:.4f} | Peak VRAM={vram_peak:.2f}GB | NaN skips={nan_skips} ===")

        if args.smoke_test:
            sec_per_1k = (pure_train_sec / args.epochs) / (len(tr_ds) / 1000.0)
            est_va = last_eval_time * (full_va_len / len(va_ds)) / 60.0
            est_te = te_elapsed * (full_te_len / len(te_ds)) / 60.0
            print(f"[ETA PROJECTION] 1 val eval ≈ {est_va:.1f} min | test inference ≈ {est_te:.1f} min")
            for n_k in (60, 80, 100):
                tr_min = (sec_per_1k * n_k) / 60.0
                print(
                    f"  {n_k}k train (1 ep): train ≈ {tr_min:.1f} min | "
                    f"total (2 evals) ≈ {(tr_min + 2 * est_va + est_te) / 60:.2f} h | "
                    f"total (1 eval) ≈ {(tr_min + est_va + est_te) / 60:.2f} h"
                )
            if best_ckpt_path.exists():
                best_ckpt_path.unlink()
        else:
            va_preds = best_oof_logits.argmax(axis=1)
            print("\n--- Classification Report (Val Fold 0) ---")
            print(classification_report(va_y, va_preds, target_names=CLASSES, digits=4))

            oof_df = pd.DataFrame({
                "id": va_ids,
                "y_true": va_y,
                "y_pred": va_preds,
                "logit_human": best_oof_logits[:, 0],
                "logit_fusion": best_oof_logits[:, 1],
                "logit_ai": best_oof_logits[:, 2],
            })
            te_preds = te_logits.argmax(axis=1)
            te_out = pd.DataFrame({
                "id": te_ids,
                "y_pred": te_preds,
                "logit_human": te_logits[:, 0],
                "logit_fusion": te_logits[:, 1],
                "logit_ai": te_logits[:, 2],
            })
            oof_df.to_parquet(OUT_DIR / "qwen15b_lora_oof_fold0.parquet", index=False)
            te_out.to_parquet(OUT_DIR / "qwen15b_lora_test_preds.parquet", index=False)

            sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
            pred_map = te_out.set_index("id")["y_pred"].map(ID2LABEL)
            sub = pd.DataFrame({"id": sample_sub["id"], "label": sample_sub["id"].map(pred_map)})
            assert len(sub) == 25_000 and (sub["id"].to_numpy() == sample_sub["id"].to_numpy()).all()
            assert sub["label"].notna().all() and set(sub["label"].unique()) == set(CLASSES)
            sub.to_csv(OUT_DIR / "submission_qwen15b_lora.csv", index=False)

            print("Saved: qwen15b_lora_adapter.pt, qwen15b_lora_oof_fold0.parquet, qwen15b_lora_test_preds.parquet, submission_qwen15b_lora.csv")
            print("\n--- Test Predicted Class % ---")
            print((sub["label"].value_counts(normalize=True) * 100).round(2).reindex(CLASSES))

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_qwen_lora.py


In [2]:
!torchrun --nproc_per_node=2 --master_port=29504 /kaggle/working/train_qwen_lora.py --max_train_samples 60000 --evals_per_epoch 1 --eval_batch_size 24

W1122 22:59:45.087000 45 torch/distributed/run.py:851] 
W1122 22:59:45.087000 45 torch/distributed/run.py:851] *****************************************
W1122 22:59:45.087000 45 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1122 22:59:45.087000 45 torch/distributed/run.py:851] *****************************************
config.json: 100%|█████████████████████████████| 684/684 [00:00<00:00, 4.10MB/s]
tokenizer_config.json: 100%|███████████████| 7.23k/7.23k [00:00<00:00, 25.6MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 51.5MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 111MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 162MB/s]
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: Use